# Validate the Reference H2O Bundle

Validate required files, manifest fields, feature order, immutable version, H2O version, checksums, and golden prediction parity before any Azure registration.

**Source:** Adapted from this repository's `notebooks/h2o_mojo/02_onboard_customer_mojo.ipynb`.

In [ ]:
from pathlib import Path
import json
import os
import sys

import h2o
import numpy as np
import pandas as pd
from dotenv import load_dotenv

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "pipelines").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")
load_dotenv(WORKSHOP_ROOT / ".env", override=True)

bundle_value = Path(os.environ["H2O_BUNDLE_DIR"])
BUNDLE_DIR = bundle_value if bundle_value.is_absolute() else WORKSHOP_ROOT / bundle_value
sys.path.insert(0, str(WORKSHOP_ROOT / "src/h2o"))
from validate_bundle import validate_bundle

summary = validate_bundle(BUNDLE_DIR, os.environ["H2O_VERSION"])
display(summary)
manifest = json.loads((BUNDLE_DIR / "model_manifest.json").read_text(encoding="utf-8"))
golden_input = pd.read_csv(BUNDLE_DIR / "golden_input.csv")
golden_expected = pd.read_csv(BUNDLE_DIR / "golden_expected.csv")

try:
    h2o.init(max_mem_size="2G", nthreads=-1)
    model = h2o.load_model(str(BUNDLE_DIR / manifest["model_file"]))
    frame = h2o.H2OFrame(golden_input)
    for column in manifest["categorical_features"]:
        frame[column] = frame[column].asfactor()
    actual = model.predict(frame).as_data_frame()
    np.testing.assert_allclose(golden_expected["predict"], actual["predict"], rtol=1e-6, atol=1e-6)
    comparison = pd.DataFrame({"expected": golden_expected["predict"], "actual": actual["predict"]})
    display(comparison.head(10))
    print("Bundle validation and golden prediction parity passed.")
finally:
    if h2o.connection() is not None:
        h2o.cluster().shutdown(prompt=False)

## Expected Result

The reference bundle passes schema, immutable-version, checksum, exact-H2O-version, feature-order, and golden-parity validation.

Next: `03_test_local_endpoint.ipynb`.